Usando playwright para extraccion y la carga

In [ ]:
from playwright.async_api import async_playwright
from pymongo import MongoClient, UpdateOne
import datetime
import random # ⬅️ Importamos la librería para generar caos matemático

# --- 1. CONEXIÓN A NOSQL ---
cliente = MongoClient('mongodb://localhost:27017/')
db = cliente['mercado_inmobiliario']
coleccion = db['departamentos_cdmx']

# Diccionario maestro de extracción: Nombre formal -> "slug" (ruta de la URL)
colonias_objetivo = {
    "Condesa": "condesa",
    "Roma Norte": "roma-norte",
    "Doctores": "doctores", 
    "Del Valle": "del-valle",
    "Narvarte": "narvarte",
    "Polanco": "polanco",
    "Coyoacán": "coyoacan"
}

print("🚀 Iniciando motor asíncrono y enlazando a MongoDB...")
playwright = await async_playwright().start()
browser = await playwright.firefox.launch(headless=False)
page = await browser.new_page()

# --- 2. BUCLE MULTI-COLONIA ---
for nombre_colonia, slug in colonias_objetivo.items():
    url = f"https://www.inmuebles24.com/departamentos-en-renta-en-{slug}.html"
    
    print("\n" + "="*50)
    print(f"📍 Misión en curso: {nombre_colonia}")
    print(f"🔗 Navegando a: {url}")
    
    try:
        await page.goto(url)
        
        # ⬅️ TÁCTICA DE EVASIÓN: Tiempo aleatorio entre 15 y 30 segundos
        tiempo_pausa = random.randint(15000, 30000) 
        print(f"⏳ Pausa táctica de {tiempo_pausa / 1000}s para imitar comportamiento humano...")
        await page.wait_for_timeout(tiempo_pausa)
        
        # ⬅️ TÁCTICA DE EVASIÓN: Simulamos scroll con el ratón
        print("🖱️ Haciendo scroll para cargar imágenes y engañar al radar...")
        await page.mouse.wheel(0, 800)
        await page.wait_for_timeout(2000) # 2 segundos viendo la página tras el scroll
        
        print("🕵️‍♂️ Ensamblando tarjetas...")
        tarjetas = await page.locator("div[class*='postingCard'], div[class*='PostingCard']").all()
        
        registros_extraidos = []
        
        for tarjeta in tarjetas[:15]: # Extraemos los primeros 15 de cada colonia
            try:
                precio_locator = tarjeta.locator("div[class*='postingPrices-module__price-container']")
                precio_raw = await precio_locator.first.inner_text() if await precio_locator.count() > 0 else "Sin precio"
                
                h2_elementos = await tarjeta.locator("h2").all()
                titulo_raw = "Sin título"
                
                for h2 in h2_elementos:
                    texto = await h2.inner_text()
                    if "MN" not in texto:
                        titulo_raw = texto
                        break
                
                documento = {
                    "titulo": titulo_raw.strip()[:100] + "...", 
                    "precio_raw": precio_raw.strip(),
                    "colonia": nombre_colonia, 
                    "fecha_extraccion": datetime.datetime.now().strftime("%Y-%m-%d")
                }
                
                if documento["precio_raw"] != "Sin precio" and documento["titulo"] != "Sin título":
                    registros_extraidos.append(documento)
                    
            except Exception as e:
                continue

        print(f"✅ Se ensamblaron {len(registros_extraidos)} diccionarios de {nombre_colonia}.")

        # --- 3. INGESTA NOSQL POR LOTE ---
        if registros_extraidos:
            operaciones = []
            for reg in registros_extraidos:
                operaciones.append(UpdateOne(
                    {"titulo": reg["titulo"], "colonia": reg["colonia"]}, 
                    {"$set": reg}, 
                    upsert=True    
                ))
                
            resultado = coleccion.bulk_write(operaciones)
            print(f"💾 Upserts aplicados en {nombre_colonia}: {resultado.upserted_count + resultado.modified_count}")
        else:
            print(f"⚠️ No se extrajeron registros de {nombre_colonia}.")
            
    except Exception as e:
        print(f"❌ Error al procesar la colonia {nombre_colonia}: {e}")

# --- 4. CIERRE DE SESIÓN ---
print("\n" + "="*50)
print(f"📊 Total histórico GLOBAL en MongoDB: {coleccion.count_documents({})}")
await browser.close()
await playwright.stop()
print("🏁 Ejecución finalizada.")

🚀 Iniciando motor asíncrono y enlazando a MongoDB...

📍 Misión en curso: Condesa
🔗 Navegando a: https://www.inmuebles24.com/departamentos-en-renta-en-condesa.html
⏳ Esperando 10s para estabilizar el DOM y evadir Cloudflare...
🕵️‍♂️ Ensamblando tarjetas...
✅ Se ensamblaron 8 diccionarios de Condesa.
💾 Upserts aplicados en Condesa: 2

📍 Misión en curso: Roma Norte
🔗 Navegando a: https://www.inmuebles24.com/departamentos-en-renta-en-roma-norte.html
⏳ Esperando 10s para estabilizar el DOM y evadir Cloudflare...
🕵️‍♂️ Ensamblando tarjetas...
✅ Se ensamblaron 0 diccionarios de Roma Norte.
⚠️ No se extrajeron registros de Roma Norte.

📍 Misión en curso: Doctores
🔗 Navegando a: https://www.inmuebles24.com/departamentos-en-renta-en-doctores.html
⏳ Esperando 10s para estabilizar el DOM y evadir Cloudflare...
🕵️‍♂️ Ensamblando tarjetas...
✅ Se ensamblaron 0 diccionarios de Doctores.
⚠️ No se extrajeron registros de Doctores.

📍 Misión en curso: Del Valle
🔗 Navegando a: https://www.inmuebles24.com

CancelledError: 

Verificacion - este bloque se conecta a MongoDB para imprimir los titulos y precios guardados

In [9]:
from pymongo import MongoClient

# Conectamos al servidor
cliente = MongoClient('mongodb://localhost:27017/')
coleccion = cliente['mercado_inmobiliario']['departamentos_cdmx']

print("🗄️ Abriendo la bóveda de MongoDB...\n")

# Traemos todos los documentos guardados
departamentos = coleccion.find()

for depa in departamentos:
    print(f"ID Interno Mongo: {depa['_id']}")
    print(f"Título: {depa['titulo']}")
    print(f"Precio: {depa['precio_raw']}")
    print("-" * 50)

🗄️ Abriendo la bóveda de MongoDB...

ID Interno Mongo: 6abddea3d5f3bb0bdafa7e19
Título: Renta de departamentos nuevos en Insurgentes Roma Sur. En una de las zonas más conectadas de la Ciud...
Precio: MN 25,150
--------------------------------------------------
ID Interno Mongo: 6abddea3d5f3bb0bdafa7e1a
Título: Sin título...
Precio: MN 27,500
--------------------------------------------------
ID Interno Mongo: 6abddea3d5f3bb0bdafa7e1b
Título: Con una ubicación privilegiada, cerca de restaurantes, bares, cafeterías, centros comerciales y vías...
Precio: MN 39,500
--------------------------------------------------
ID Interno Mongo: 6abebaca151ced0232d8216f
Título: ¡Bienvenido al paraíso urbano en el corazón de la Condesa! Este departamento amueblado de 80 m², ori...
Precio: MN 27,500
--------------------------------------------------
ID Interno Mongo: 6abebaca151ced0232d82170
Título: Residencia con jardín Y roof top - se entrega en octubre. Una excelente opción tanto para vivir como...
Pr

Tranformacion de datos usando pandas

In [ ]:
import pandas as pd
from pymongo import MongoClient

print(' Conectando a MongoDB y extrayendo datos para DataFrame...')
cliente = MongoClient('mongodb://localhost:27017/')
coleccion = cliente['mercado_inmobiliario']['departamentos_cdmx']

# 1. Extraemos todo y lo convertimos en una lista de diccionarios
datos_crudos = list(coleccion.find())

# 2. Inyectamos la lista directamente en un DataFrame de Pandas
df = pd.DataFrame(datos_crudos)

# Eliminamos la columna '_id' interna de Mongo porque no la necesitamos para análisis
if '_id' in df.columns:
    df = df.drop(columns=['_id'])

print('DataFrame original (sucio)')
display(df.head()) # 'display' renderiza una tabla bonita en Jupyterdisplay' renderiza una tabla bonita en Jupyter

print("\n Limpiando y normalizando columnas...")

# 3. Limpieza de la columna de precio
# Reemplazamos "MN", espacios en blanco y comas por nada ("")
df['precio_numerico'] = df['precio_raw'].str.replace('MN', '', regex=False)
df['precio_numerico'] = df['precio_numerico'].str.replace(',', '', regex=False)
df['precio_numerico'] = df['precio_numerico'].str.strip()

# 4. Cast (Conversión) de texto a número flotante
# errors='coerce' forzará a NaN (Not a Number) cualquier cosa que no se pueda convertir
df['precio_numerico'] = pd.to_numeric(df['precio_numerico'], errors='coerce')

print('DataFrame limpio y listo para matematicas:')
display(df[['titulo', 'colonia', 'precio_numerico']].head())

# Validamos los tipos de datos para confirmar que 'precio_numerico' ya es float64
print("\n Tipos de datos actuales:")
print(df.dtypes)




 Conectando a MongoDB y extrayendo datos para DataFrame...
DataFrame original (sucio)


,colonia,titulo,fecha_extraccion,precio_raw
0,Condesa,Renta de departamentos nuevos en Insurgentes R...,2026-09-30,"MN 25,150"
1,Condesa,Sin título...,2026-10-01,"MN 27,500"
2,Condesa,"Con una ubicación privilegiada, cerca de resta...",2026-09-30,"MN 39,500"
3,Condesa,¡Bienvenido al paraíso urbano en el corazón de...,2026-10-01,"MN 27,500"
4,Condesa,Residencia con jardín Y roof top - se entrega ...,2026-10-01,"MN 67,000"



 Limpiando y normalizando columnas...
DataFrame limpio y listo para matematicas:


,titulo,colonia,precio_numerico
0,Renta de departamentos nuevos en Insurgentes R...,Condesa,25150
1,Sin título...,Condesa,27500
2,"Con una ubicación privilegiada, cerca de resta...",Condesa,39500
3,¡Bienvenido al paraíso urbano en el corazón de...,Condesa,27500
4,Residencia con jardín Y roof top - se entrega ...,Condesa,67000



 Tipos de datos actuales:
colonia               str
titulo                str
fecha_extraccion      str
precio_raw            str
precio_numerico     int64
dtype: object
